# 01 - Exploratory Data Analysis

Objetivo: describir clientes, transacciones, movimientos, temporalidad, concentracion y calidad mediante tablas y visualizaciones Plotly.

In [ ]:
import sys
from pathlib import Path
import polars as pl
import plotly.express as px
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, load_catalogs

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
transactions = load_transactions(ROOT / "transacciones_sinteticas.csv")
types, _ = load_catalogs(ROOT / "catalogo_tipos_subtipos.csv", ROOT / "catalogo_bancos.csv")
transactions = transactions.with_columns(pl.col("fecha_transaccion").dt.date().alias("date"), pl.col("fecha_transaccion").dt.month().alias("month"))
transactions.head()

In [ ]:
customer_activity = transactions.group_by("codigo_cliente").agg(pl.len().alias("transactions"), pl.col("monto").sum().alias("net_amount"))
type_frequency = transactions.group_by("tipo_transaccion").agg(pl.len().alias("transactions"), pl.col("monto").sum().alias("net_amount")).join(types.group_by("tipo_transaccion").agg(pl.col("descripcion_tipo").first()), on="tipo_transaccion")
movement_summary = transactions.group_by("tipo_movimiento").agg(pl.len().alias("transactions"), pl.col("monto").mean().alias("mean_amount"), pl.col("monto").sum().alias("total_amount"))
monthly = transactions.group_by("month").agg(pl.len().alias("transactions"), pl.col("monto").sum().alias("net_amount")).sort("month")
customer_activity.describe(), movement_summary, monthly

In [ ]:
fig = px.histogram(customer_activity.to_dicts(), x="transactions", nbins=50, title="Distribucion de transacciones por cliente")
fig.show()
fig = px.bar(type_frequency.sort("transactions", descending=True).head(20).to_dicts(), x="descripcion_tipo", y="transactions", title="20 tipos mas frecuentes")
fig.show()
fig = px.line(monthly.to_dicts(), x="month", y="transactions", markers=True, title="Evolucion mensual")
fig.show()

## Interpretacion
Las tablas y figuras son descriptivas; no imponen segmentos. La siguiente etapa analiza valores extremos sin eliminarlos automaticamente.